# h5_report

이미지가 입혀진 데이터셋 `.h5`(`h5_add_images` 출력)에서 **랜덤 N개 에피소드**를 뽑아, 각 에피소드의 **메타데이터 표 + 인라인 필름스트립(PNG) + MP4 링크**를 담은 Markdown 리포트를 만든다. 숫자로는 안 보이는 (이미지 + 지시 + 행동) 정렬을 사람이 눈으로 점검하는 **학습 전 가드레일**.

시뮬레이터는 안 돈다 — `.h5`에 기록된 RGB 프레임만 읽어 조립 (`scripts/h5_to_media.py`의 렌더러 `card`/`video` 재사용). 지시문/라벨은 데이터셋 `.json`의 `label_metadata`(env가 선언)에서 디코드하므로 **task-무관**.

**`run`** — 리포트 생성

| 파라미터 | 설명 | 기본값 |
|---|---|---|
| `traj_path` | 데이터셋 .h5 경로 (h5_add_images 출력) | (필수) |
| `n` | 랜덤 에피소드 수 | `3` |
| `seed` | 랜덤 시드 (지정 시 재현 가능) | `None` |
| `fps` | 영상 fps | `15` |

출력: 입력 옆 `reports/` 폴더에 `<stem>.report.md` + 에피소드별 `*.card.png` / `*.mp4`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from scripts.h5_report import run
from IPython.display import Markdown, Image, display

ds = ROOT / 'data' / 'datasets' / 'ThreeColoredCubes-v1' / 'motionplanning.rgb.pd_joint_pos.physx_cpu.h5'

report_md = run(ds, n=3, seed=42)   # 랜덤 3개 (seed로 재현). seed=None 이면 매번 랜덤.
print('report ->', report_md)

In [ ]:
# 리포트 본문(메타 표 + mp4 링크) 표시.
# (MD 안의 ![]() 이미지 링크는 reports/ 기준 상대경로라 노트북에선 깨질 수 있어
#  필름스트립은 아래에서 따로 인라인 표시한다.)
display(Markdown(report_md.read_text(encoding='utf-8')))

for png in sorted(report_md.parent.glob('*.card.png')):
    print(png.name)
    display(Image(str(png)))

CLI: `python scripts/h5_report.py --traj-path <h5> --n 3 [--seed 42]`

보통 흐름: `h5_add_images` 로 데이터셋을 만든 뒤 그 출력 `.h5` 를 여기에 넣어 품질을 점검한다.